# Legacy 특징 추출

원본 `최종.ipynb`의 15개 URL 특징 추출 코드를 분리한 노트북입니다. 특징 계산과 `-1/0/1` 인코딩은 원본과 동일합니다.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.cluster import KMeans
# from sklearn.cluster import DBSCAN
from sklearn.cluster import MeanShift
from sklearn.mixture import GaussianMixture
from sklearn.cluster import AgglomerativeClustering
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from scipy.spatial.distance import euclidean
from threading import Thread


import math
import tkinter as tk
from tkinter import ttk
from PIL import Image, ImageTk  # PIL 라이브러리에서 이미지 로드
from tkinter import PhotoImage
from PIL import Image, ImageTk
from tkinter import messagebox

import random
import time
import re # 정규 표현식
import requests # HTTP 요청 및 응답을 받아오는 모듈
import whois # 도메인의 등록 기간을 파악하는 모듈
import sys # 파이썬 인터프리터의 상태를 확인하는 모듈
import socket
import time
import tld
from urllib.parse import urlparse # URL을 파싱하는 모듈
from urllib.request import urlopen, Request # URL을 열고 읽는 모듈
from bs4 import BeautifulSoup, SoupStrainer # HTML 문서를 파싱하는 모듈
from tld import get_tld # URL에서 도메인을 추출하는 모듈
from tld.exceptions import TldBadUrl, TldDomainNotFound # 도메인 추출 시 예외 처리 모듈
from datetime import datetime, timedelta # 날짜와 시간을 다루는 모듈

In [ ]:
from pathlib import Path


def find_project_root(start):
    """현재 실행 위치에서 프로젝트 루트를 찾습니다."""
    current = Path(start).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "legacy" / "notebooks" / "최종.ipynb").exists():
            return candidate
    raise FileNotFoundError("프로젝트 루트를 찾을 수 없습니다.")


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_PATH = PROJECT_ROOT / "data" / "legacy" / "전체15.csv"
MODEL_DIR = PROJECT_ROOT / "legacy" / "models"
IMAGE_DIR = PROJECT_ROOT / "legacy" / "images"


## 15개 특징 함수


In [ ]:
# 1. URL에 IP가 포함된 경우
def having_ip_address(url):
    ipv4_pattern = re.compile(r'^https?://(\d{1,3}\.){3}\d{1,3}(:\d+)?(/|$)')
    hex_ipv4_pattern = re.compile(r'^https?://0x([0-9a-fA-F]{1,2})\.(0x[0-9a-fA-F]{1,2})\.(0x[0-9a-fA-F]{1,2})\.(0x[0-9a-fA-F]{1,2})(:\d+)?(/|$)')
    ipv6_pattern = re.compile(r'^https?://([0-9a-fA-f:]+)(:\d+)?(/|$)')

    if ipv4_pattern.match(url) or hex_ipv4_pattern.match(url) or ipv6_pattern.match(url):
        return -1
    else:
        return 1

# 2. 긴 URL 사용
def url_length(url):
    if len(url) < 54:
        return 1
    elif len(url) >= 54 and len(url) <= 75:
        return 0
    else:
        return -1

# 3. URL 단축 서비스 사용
shorteners = ['bit.ly', 'kl.am', 'cli.gs', 'bc.vc', 'po.st', 'v.gd', 'bkite.com', 'shorl.com', 'scrnch.me', 'to.ly', 'adf.ly', 'x.co', '1url.com', 'ad.vu', 'migre.me', 'su.pr', 'smallurl.co', 'cutt.us', 'filoops.info', 'shor7.com', 'yfrog.com', 'tinyurl.com', 'u.to', 'ow.ly', 'ff.im', 'rubyurl.com', 'r2me.com', 'post.ly', 'twitthis.com', 'buzurl.com', 'cur.lv', 'tr.im', 'bl.lnk', 'tiny.cc', 'lnkd.in', 'q.gs', 'is.gd', 'hurl.ws', 'om.ly', 'prettylinkpro.com', 'qr.net', 'qr.ae', 'snipurl.com', 'ity.im', 't.co', 'db.tt', 'link.zip.net', 'doiop.com', 'url4.eu', 'poprl.com', 'tweez.me', 'short.ie', 'me2.do', 'bit.do', 'shorte.st', 'go2l.ink', 'yourls.org', 'wp.me', 'goo.gl', 'j.mp', 'twurl.nl', 'snipr.com', 'shortto.com', 'vzturl.com', 'u.bb', 'shorturl.at', 'han.gl', 'wo.gl', 'wa.gl']

def shortening_service(url):
    domain = urlparse(url).netloc
    if domain in shorteners:
        return -1
    return 1

# 4. URL에 '@' 기호 포함
def having_at_symbol(url):
    if '@' in url or '%40' in url:
        return -1
    else:
        return 1

# 5. '//'를 사용한 리다이렉션
def double_slash_redirecting(url):
    if '//' in url[7:]:
        return -1
    else:
        return 1

# 6. URL의 접두사/접미사에 '-'가 포함된 경우
def prefix_suffix(url):
    # ip형식일 경우
    if having_ip_address(url) == -1:
        return 0
    try:
        domain = get_tld(url, as_object=True)

        if '-' in domain.domain:
            return -1
        elif '-' in domain.subdomain:
            return -1
        else:
            return 1
    except tld.exceptions.TldDomainNotFound:
        return -1

# 7. 서브도메인이 있는지 확인
def having_sub_domain(url):
    if having_ip_address(url) == -1:
        return 0
    # www. 제거
    if 'www.' in url[:12]:
        url = url.replace('www.', '')

    try:
        domain = get_tld(url, as_object=True)

        if domain.subdomain == '': # 서브 도메인이 없을 경우
            return 1
        dot = domain.subdomain.count('.')
        if dot == 0: # 서브 도메인이 있는 경우 .이 없으면 의심 1개 이상이면 피싱
            return 0
        else:
            return -1
    except tld.exceptions.TldDomainNotFound:
        return -1

# 8. 도메인 등록 기간
def get_total_date(url):
    domain = whois.whois(url)
    expiration_date = domain.expiration_date
    if isinstance(expiration_date, list):
        expiration_date = expiration_date[0]
    updated_date = domain.updated_date
    if isinstance(updated_date, list):
        updated_date = updated_date[0]
    if expiration_date is None or updated_date is None:
        return None
    total_date = (expiration_date - updated_date).days
    return total_date

def domain_registration_length(url):
    try:
        total_date = get_total_date(url)
        if total_date is None:
            return 0
        if total_date <= 365:
            return -1
        else:
            return 1
    except whois.parser.PywhoisError:
        return -1
    except:
        return 0

# 9. Favicon
def favicon(url):
    try:
        response = requests.get(url)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        parsed_url = urlparse(url)
        base_domain = parsed_url.netloc
        favicon_tags = soup.find_all('link', rel=lambda value: value and 'icon' in value.lower())
        if not favicon_tags:
            return 1
        for tag in favicon_tags:
            href = tag.get('href')
            if href:
                favicon_domain = urlparse(href).netloc
                if favicon_domain == '':
                    return 1
                else:
                    return -1
        return 1
    except:
        return -1

# 10. 비표준 포트 사용
def remove_schemes(url):
    if 'http://' in url:
        url = url.replace('http://', '')
    elif 'https://' in url:
        url = url.replace('https://', '')
    else:
        return url
    return url

def port(url):
    domain = remove_schemes(url)
    try:
        ip = socket.gethostbyname(domain)
    except:
        return -1
    socket.setdefaulttimeout(2)
    ports = [80, 21, 22, 23, 445, 1433, 1521, 3306, 3389]
    for port in ports:
        s = socket.socket()
        if port == 80:
            try:
                s.connect((ip, port))
                s.close()
            except:
                return -1
        else:
            try:
                s.connect((ip, port))
                s.close()
                return -1
            except:
                pass
    return 1

# 11. 도메인 부분에 HTTPS 존재 여부
def https_token(url):
    try:
        domain = url.split('/')[2]
    except IndexError:
        return 1
    if "https" in domain:
        return -1
    else:
        return 1

# 12. 도메인 수명(6개월 미만)
def age_of_domain(url):
    try:
        domain_info = whois.whois(url)
        expiration_date = domain_info.expiration_date
        if isinstance(expiration_date, list):
            expiration_date = expiration_date[0]
        current_date = datetime.now()
        remain_date = expiration_date - current_date
        if remain_date >= timedelta(days=182):
            return 1
        else:
            return -1
    except:
        return 0

# 13. DNS 기록(유/무)
def dns_record(url):
    try:
        whois.whois(url)
    except:
        return -1
    return 1

# 14. 리다이렉션 횟수(피싱 최대 4번)
def count_redirection(url):
    try:
        count = 0
        res = requests.head(url, allow_redirects=True, timeout=5)
        for resp in res.history:
            if resp.status_code in [301, 302]:
                count += 1
        if count <= 1:
            return 1
        elif 2 <= count < 4:
            return 0
        else:
            return -1
    except:
        return 0

# 15. 우클릭 금지
def disabling_right_click(url):
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'
        }
        res = requests.get(url, timeout=5, headers=headers)
        if 'event.button==2' in res.text:
            return -1
        else:
            return 1
    except:
        return 0

## 통합 특징 추출 함수


In [ ]:
def feature_extract(url):
    features = {
        "having_ip_address": having_ip_address(url),
        "url_length": url_length(url),
        "shortening_service": shortening_service(url),
        "having_at_symbol": having_at_symbol(url),
        "double_slash_redirecting": double_slash_redirecting(url),
        "prefix_suffix": prefix_suffix(url),
        "having_sub_domain": having_sub_domain(url),
        "domain_registration_length": domain_registration_length(url),
        "favicon": favicon(url),
        "port": port(url),
        "https_token": https_token(url),
        "age_of_domain": age_of_domain(url),
        "dns_record": dns_record(url),
        "count_redirection": count_redirection(url),
        "disabling_right_click": disabling_right_click(url)
    }

    # 특성 값을 DataFrame으로 변환
    features_df = pd.DataFrame([features])
    # NumPy 배열로 변환
    features_array = features_df.values.flatten()

    return features_df, features_array
